# T4 state-free text embeddings (TRAIN + CAL pool texts)
Same procedure as `tools/capture_textemb_v3.py` (3050 reference): 'Next action:' + text, depths 18/24/30, mean over the text span, fp16 storage, shards of 2048, registered gate cosine >= 0.9999 batch-vs-alone (run on EVERY shard here, stricter than registered).
Device lock as the T4 state capture: Tesla T4, fp16, torch 2.10.0+cu128, transformers 5.17.0, revision 768f209d9e. Contains NO LOCKED data. Run with Save & Run All, accelerator GPU T4 x2 (one GPU is used), Internet ON, the private dataset attached.

In [ ]:
import subprocess, sys, json, os, glob, shutil, hashlib, time, tarfile
import torch
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv", shell=True, capture_output=True, text=True).stdout)
assert all("T4" in torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))
print("kernel torch", torch.__version__)
# device lock (amendment e1): the WORKER process must run torch 2.10.0+cu128. Kaggle ignores 'Pin to original environment' for committed runs, so install it for the worker.
if torch.__version__ != "2.10.0+cu128":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.10.0", "--index-url", "https://download.pytorch.org/whl/cu128", "--extra-index-url", "https://pypi.org/simple"], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-900:])
    r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], capture_output=True, text=True); print(r.stdout[-300:], r.stderr[-300:])
wt = subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__)"], capture_output=True, text=True).stdout.strip()
print("worker torch", wt); assert wt == "2.10.0+cu128", f"device lock (amendment e1): worker torch must be 2.10.0+cu128, got {wt}"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.17.0", "huggingface_hub"], capture_output=True, text=True); print(r.stdout[-300:], r.stderr[-400:])
import transformers; assert transformers.__version__ == "5.17.0"; print("transformers", transformers.__version__)
chk = subprocess.run([sys.executable, "-c", "import torch, transformers; from transformers import Qwen3ForCausalLM, AutoTokenizer; print(torch.__version__, transformers.__version__)"], capture_output=True, text=True)
print("worker preflight:", chk.stdout.strip(), chk.stderr[-1500:]); assert chk.returncode == 0 and chk.stdout.split()[0] == "2.10.0+cu128", "worker preflight failed (device lock e1)"

In [ ]:
hits = glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True); assert hits, "attach the private dataset"
SRC = os.path.dirname(hits[0]); W = "/kaggle/temp/proj"; shutil.copytree(SRC, W, dirs_exist_ok=True)
man = json.load(open(W + "/MANIFEST.json"))
bad = [f["path"] for f in man["files"] if hashlib.sha256(open(f"{W}/{f['path']}", "rb").read()).hexdigest() != f["sha256"]]; assert not bad, bad
assert man["contains_locked"] is False and man["contains_trajectories"] is False and man["contains_labels"] is False
texts = json.load(open(W + "/data/texts_trainval.json")); assert hashlib.sha256(json.dumps(texts).encode()).hexdigest() == man["texts_sha256"] and len(texts) == man["texts"]
print("bundle OK:", len(man["files"]), "files,", len(texts), "texts, source commit", man["source_commit"][:8])

In [ ]:
from huggingface_hub import snapshot_download
rev = json.load(open(W + "/results/raw/qwen3-4b-thinking-2507.manifest.json"))
MODEL = snapshot_download(rev["repo"], revision=rev["revision"], local_dir="/kaggle/temp/qwen3-4b")
for f in rev["files"]:
    if f["sha256"]:
        h = hashlib.sha256()
        with open(f"{MODEL}/{f['name']}", "rb") as fh:
            for c in iter(lambda: fh.read(1 << 24), b""): h.update(c)
        assert h.hexdigest() == f["sha256"], f["name"]
print("model OK at revision", rev["revision"][:10])

In [ ]:
OUT = "/kaggle/working/textemb_t4"
env = dict(os.environ, PYTHONPATH=W, CUDA_VISIBLE_DEVICES="0", PYTORCH_ALLOC_CONF="expandable_segments:True")
log = open("/kaggle/working/textemb.out", "w")
t0 = time.time()
p = subprocess.Popen([sys.executable, f"{W}/tools/capture_textemb_v3_t4.py", "--texts", f"{W}/data/texts_trainval.json", "--outdir", OUT, "--model", MODEL, "--dtype", "fp16", "--require_device", "T4"], cwd=W, env=env, stdout=log, stderr=subprocess.STDOUT)
try:
    while p.poll() is None:
        time.sleep(60)
        print(f"[textemb] {len(glob.glob(OUT + '/shard-*.pt'))} shards, {(time.time() - t0) / 60:.1f} min", flush=True)
    L = open("/kaggle/working/textemb.out").readlines()
    print("exit", p.returncode, "\n" + "".join(L[:25] + [" ...\n"] + L[-25:]) if p.returncode else "\n" + "".join(L[-8:]))
    assert p.returncode == 0, "the embedding run failed (a failed gate aborts it); see the tail above"
finally:
    if p.poll() is None: p.terminate()
    if os.path.isdir(OUT):
        with tarfile.open("/kaggle/working/textemb_t4.tar", "w") as t: t.add(OUT, arcname="textemb_t4")
    print(subprocess.run("ls -la /kaggle/working; du -sh /kaggle/working", shell=True, capture_output=True, text=True).stdout)

In [ ]:
import statistics
dev = json.load(open(OUT + "/textemb-device.json"))
assert dev["torch"] == "2.10.0+cu128" and dev["transformers"] == "5.17.0", dev
shards = sorted(glob.glob(OUT + "/shard-*.pt")); n, ents = 0, []
for f in shards:
    d = torch.load(f); n += len(d)
    assert all(v.shape == (3, 2560) and v.dtype == torch.float16 and torch.isfinite(v).all() for v in d.values())
    ents.append(dict(file=os.path.basename(f), texts=len(d), sha256=hashlib.sha256(open(f, "rb").read()).hexdigest()))
assert n == len(texts), (n, len(texts))
last = [l for l in open("/kaggle/working/textemb.out") if l.startswith("DONE")]
json.dump(dict(device=dev, texts=n, shards=ents, done_line=last[-1].strip() if last else None, wall_minutes=(time.time() - t0) / 60), open("/kaggle/working/textemb_manifest.json", "w"), indent=1)
print(json.dumps(dict(device=dev, texts=n, shards=len(shards), done=last[-1].strip() if last else None), indent=1))